# Практика · OpenCV: базові операції

> Лекція: [lecture.html](lecture.html) · Домашнє: [homework.html](homework.html) · Тест: [quiz.html](quiz.html)

**Мережа не потрібна:** усі зображення ми малюємо самі, формулами. Тому числа в тебе на
екрані будуть точнісінько такі самі, як у лекції.

Що зробимо:

1. згенеруємо те саме «фото товару з оголошення», що й у темі 01;
2. запишемо його на диск і прочитаємо назад — PNG проти JPEG;
3. подивимось, що `imread` повертає на неіснуючому шляху, і чому це `None`, а не виняток;
4. переконаємось, що зріз ділить памʼять з оригіналом, а `.copy()` — ні;
5. звіримо власну обрізку зрізом із `cv2.getRectSubPix`;
6. поміряємо, скільки шуму лишається після трьох інтерполяцій;
7. звіримо власне дзеркалення з `cv2.flip`;
8. порахуємо, скільки пікселів зникає при повороті, і побудуємо полотно, у якому не зникає нічого;
9. подивимось на чотири межові режими числами;
10. зробимо маску індикатора через HSV і намалюємо по ній рамку детектора;
11. порівняємо глобальний поріг, Оцу й адаптивний на нерівно освітленій наліпці;
12. поміряємо час усіх операцій на кадрі 1080p і порівняємо з бюджетом 25 кадрів за секунду.

## 0 · Що нам знадобиться

Три бібліотеки, ті самі, що й у темі 01. `tempfile` потрібен, щоб файли, які ми запишемо,
лягли в тимчасову теку й прибрались у кінці — репозиторій лишиться чистим.

In [ ]:
import os
import shutil
import tempfile
import time

import numpy as np
import cv2
import matplotlib.pyplot as plt

%matplotlib inline

# усі файли цього зошита живуть тут і будуть видалені в останній клітинці
WORK_DIR = tempfile.mkdtemp(prefix="opencv_basics_")

print("numpy     ", np.__version__)
print("opencv    ", cv2.__version__)
print("тимчасова тека:", WORK_DIR)

## 1 · Те саме фото товару, що й у темі 01

Функція нижче — дослівно та сама, що будувала знімок у попередній темі: фон, тінь, корпус
зі скругленими кутами, екран із заставкою, зелений індикатор, відблиск і шум матриці.
Шум детермінований, тому масив у всіх читачів побайтово однаковий.

Зверни увагу на останній рядок: ми одразу робимо **дві** версії. `photo_rgb` — у порядку
каналів R, G, B (його чекає matplotlib), `photo_bgr` — у порядку B, G, R (його чекає
OpenCV). Пастка BGR розібрана в темі 01; тут ми просто тримаємо обидві й не плутаємось.

In [ ]:
IMAGE_HEIGHT, IMAGE_WIDTH = 240, 320


def rounded_gap(row_grid, col_grid, left, top, right, bottom, radius):
    """Квадрат відстані від пікселя до скругленого прямокутника.

    Усередині прямокутника дає 0, тому одна й та сама функція годиться
    і щоб заповнити фігуру, і щоб намалювати мʼяку тінь навколо неї.
    """
    gap_x = np.maximum(np.maximum(left + radius - col_grid,
                                  col_grid - (right - radius)), 0.0)
    gap_y = np.maximum(np.maximum(top + radius - row_grid,
                                  row_grid - (bottom - radius)), 0.0)
    return gap_x * gap_x + gap_y * gap_y


def sensor_noise(height, width):
    """Детермінований «шум матриці»: значення визначається номером елемента,
    тому знімок побайтово однаковий у всіх середовищах."""
    index = np.arange(height * width * 3, dtype=np.int64)
    value = (index + 1) * 16807 % 2147483647
    value = value ^ (value >> 13)
    value = value * 48271 % 2147483647
    value = value ^ (value >> 17)
    value = value * 16807 % 2147483647
    return (value % 11 - 5).reshape(height, width, 3).astype(np.float64)


def make_phone_photo():
    """Синтетичне фото телефона: масив (240, 320, 3) типу uint8, порядок каналів RGB."""
    row_grid, col_grid = np.mgrid[0:IMAGE_HEIGHT, 0:IMAGE_WIDTH].astype(np.float64)
    down = row_grid / IMAGE_HEIGHT
    right = col_grid / IMAGE_WIDTH

    photo = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH, 3), dtype=np.float64)

    # стільниця: тепла бежева поверхня, трохи темніша знизу
    photo[:, :, 0] = 214.0 - 26.0 * down + 10.0 * right
    photo[:, :, 1] = 201.0 - 24.0 * down + 8.0 * right
    photo[:, :, 2] = 182.0 - 20.0 * down + 6.0 * right

    # тінь: силует корпусу, зсунутий вниз-вправо і розмитий по відстані
    distance = np.sqrt(rounded_gap(row_grid, col_grid, 107, 37, 235, 229, 18.0)) - 18.0
    darkness = np.clip((18.0 - distance) / 18.0, 0.0, 1.0)
    photo *= (1.0 - 0.30 * darkness)[:, :, None]

    # корпус телефона
    body = rounded_gap(row_grid, col_grid, 96, 24, 224, 216, 18.0) <= 18.0 ** 2
    sheen = 10.0 * (1.0 - down)
    photo[:, :, 0] = np.where(body, 58.0 + sheen, photo[:, :, 0])
    photo[:, :, 1] = np.where(body, 64.0 + sheen, photo[:, :, 1])
    photo[:, :, 2] = np.where(body, 74.0 + sheen, photo[:, :, 2])

    # екран: заставка з діагональним переходом від синього до помаранчевого
    screen = rounded_gap(row_grid, col_grid, 105, 33, 215, 207, 6.0) <= 6.0 ** 2
    ramp = ((col_grid - 105.0) / 110.0 + (row_grid - 33.0) / 174.0) / 2.0
    photo[:, :, 0] = np.where(screen, 26.0 + 206.0 * ramp, photo[:, :, 0])
    photo[:, :, 1] = np.where(screen, 58.0 + 66.0 * ramp, photo[:, :, 1])
    photo[:, :, 2] = np.where(screen, 170.0 - 128.0 * ramp, photo[:, :, 2])

    # зелена смужка індикатора на екрані
    indicator = rounded_gap(row_grid, col_grid, 117, 170, 203, 186, 5.0) <= 5.0 ** 2
    photo[:, :, 0] = np.where(indicator, 40.0, photo[:, :, 0])
    photo[:, :, 1] = np.where(indicator, 200.0, photo[:, :, 1])
    photo[:, :, 2] = np.where(indicator, 90.0, photo[:, :, 2])

    # відблиск на склі: світла смуга під кутом, тільки в межах екрана
    band = (col_grid - 105.0) * 0.80 + (row_grid - 33.0) * 0.55
    glare = np.clip(1.0 - np.abs(band - 74.0) / 30.0, 0.0, 1.0)
    photo += (glare * glare * 95.0 * screen)[:, :, None]

    photo += sensor_noise(IMAGE_HEIGHT, IMAGE_WIDTH)

    return np.clip(photo, 0, 255).astype(np.uint8)


photo_rgb = make_phone_photo()
# OpenCV працює в порядку BGR — тримаємо обидві версії, щоб не плутатись далі
photo_bgr = cv2.cvtColor(photo_rgb, cv2.COLOR_RGB2BGR)

print("shape:", photo_rgb.shape, " dtype:", photo_rgb.dtype)
print("памʼяті, байтів:", photo_rgb.nbytes)

Подивимось очима. Показуємо `photo_rgb` — matplotlib чекає саме такий порядок каналів.

In [ ]:
plt.figure(figsize=(5, 3.8))
plt.imshow(photo_rgb)
plt.title("фото товару з оголошення")
plt.axis("off")
plt.show()

print("це синтетичне зображення, згенероване формулами вище")

## 2 · Запис і читання: що викидає JPEG

Запишемо той самий масив двома форматами й прочитаємо назад. Цікавить не тільки розмір
файлу, а й **чи збігається прочитаний масив із записаним**.

In [ ]:
png_path = os.path.join(WORK_DIR, "phone.png")
jpg_path = os.path.join(WORK_DIR, "phone.jpg")

# imwrite чекає BGR — саме тому ми тримаємо photo_bgr
cv2.imwrite(png_path, photo_bgr)
cv2.imwrite(jpg_path, photo_bgr, [cv2.IMWRITE_JPEG_QUALITY, 90])

png_back = cv2.imread(png_path)
jpg_back = cv2.imread(jpg_path)

# порівнюємо в int, бо різниця двох uint8 переповнилась би (пастка з теми 01)
jpeg_diff = np.abs(jpg_back.astype(np.int32) - photo_bgr.astype(np.int32))
changed_pixels = int(jpeg_diff.any(axis=2).sum())

print("сирий масив у памʼяті, байтів :", photo_bgr.nbytes)
print("PNG на диску, байтів          :", os.path.getsize(png_path))
print("JPEG на диску, байтів         :", os.path.getsize(jpg_path))
print()
print("PNG  прочитався побайтово так само:", np.array_equal(png_back, photo_bgr))
print("JPEG прочитався побайтово так само:", np.array_equal(jpg_back, photo_bgr))
print()
print("JPEG: пікселів змінилось     :", changed_pixels, "із", IMAGE_HEIGHT * IMAGE_WIDTH)
print("JPEG: середня похибка, рівнів:", round(float(jpeg_diff.mean()), 2))
print("JPEG: найбільша похибка      :", int(jpeg_diff.max()))

Файл менший у півтора десятка разів — а майже кожен піксель має інше значення. Оком цього
не видно, але задача, у якій важлива ледь помітна подряпина, може цього не пережити.

**Прапорці читання.** Перевіримо всі три на власному файлі з альфа-каналом.

In [ ]:
# PNG із прозорістю: додаємо четвертий канал, наполовину прозорий
photo_with_alpha = np.dstack([photo_bgr,
                              np.full((IMAGE_HEIGHT, IMAGE_WIDTH), 128, np.uint8)])
alpha_path = os.path.join(WORK_DIR, "phone_alpha.png")
cv2.imwrite(alpha_path, photo_with_alpha)

print("без прапорця        :", cv2.imread(alpha_path).shape, " ← прозорість викинуто мовчки")
print("IMREAD_GRAYSCALE    :", cv2.imread(alpha_path, cv2.IMREAD_GRAYSCALE).shape)
print("IMREAD_UNCHANGED    :", cv2.imread(alpha_path, cv2.IMREAD_UNCHANGED).shape, " ← альфа на місці")

## 3 · `imread` на неіснуючому файлі: чому `None`, а не виняток

Найважливіші пʼять рядків цього зошита. Ми навмисно просимо файл, якого немає.

Зверни увагу: клітинка **не падає**. У вивід (точніше, у stderr) впаде рядок зі словом
`WARN` — і все. Функція поверне `None`, а програма спокійно піде далі.

Причина в тому, що під обгорткою — код на C++, де ця функція історично повертала порожню
матрицю. Обгортка чесно переклала порожню матрицю в `None`. Наслідок для нас: помилка
проявиться пізніше й в іншому місці — на першій же спробі щось із цим `None` зробити.

In [ ]:
missing_path = os.path.join(WORK_DIR, "цього_файлу_немає.png")

result = cv2.imread(missing_path)

print("файл існує        :", os.path.exists(missing_path))
print("що повернув imread:", repr(result))
print("тип результату    :", type(result).__name__)
print()
print("клітинка не впала — саме в цьому й проблема")

А ось як це виглядає, коли `None` доживає до наступного рядка. Ловимо помилку явно, щоб
побачити її текст: саме цей `AttributeError` бачить людина, яка помилилась у шляху, — і
шукає причину зовсім не там, де вона є.

In [ ]:
try:
    print(result.shape)
except AttributeError as error:
    print("AttributeError:", error)
    print()
    print("у повідомленні немає ні слова про файл або шлях")

print()
print("правильний спосіб:")


def imread_or_die(path):
    """Читає зображення й одразу падає з осмисленим повідомленням, якщо не вийшло.

    Один рядок перевірки перетворює мовчазну поломку на гучну — саме там,
    де насправді помилка, а не через двадцять рядків.
    """
    image = cv2.imread(path)
    if image is None:
        raise FileNotFoundError(f"не вдалося прочитати зображення: {path}")
    return image


try:
    imread_or_die(missing_path)
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

## 4 · Зріз ділить памʼять, `.copy()` — ні

Тепер поміряємо те, про що йшлося в розділі 3 лекції. Вирізаємо шматок екрана й пишемо в
нього — двічі: спершу у зріз, потім у копію.

In [ ]:
ROW_FROM, ROW_TO = 40, 120
COL_FROM, COL_TO = 110, 210

# перша спроба: правимо зріз
image_one = photo_bgr.copy()
patch_view = image_one[ROW_FROM:ROW_TO, COL_FROM:COL_TO]

print("np.shares_memory(зріз, оригінал):", np.shares_memory(patch_view, image_one))
print("значення до правки             :", int(image_one[60, 150, 0]))

patch_view[:, :, 0] = 255          # «підправимо тільки шматок»

print("значення після правки          :", int(image_one[60, 150, 0]))
spoiled = int((image_one != photo_bgr).any(axis=2).sum())
print("пікселів оригіналу змінилось   :", spoiled)

In [ ]:
# друга спроба: правимо копію
image_two = photo_bgr.copy()
patch_copy = image_two[ROW_FROM:ROW_TO, COL_FROM:COL_TO].copy()

print("np.shares_memory(копія, оригінал):", np.shares_memory(patch_copy, image_two))
print("значення до правки              :", int(image_two[60, 150, 0]))

patch_copy[:, :, 0] = 255

print("значення після правки           :", int(image_two[60, 150, 0]))
print("пікселів оригіналу змінилось    :", int((image_two != photo_bgr).any(axis=2).sum()))
print()
print("ціна копії, байтів:", patch_copy.nbytes, "= висота × ширина × канали")

## 5 · Перевірка «наше = бібліотечне», раз перший: обрізка

Наш зріз проти `cv2.getRectSubPix`. Функція бібліотеки бере **центр** шматка, а не кут,
тож центр для зрізу `[y0:y0+h, x0:x0+w]` дорівнює `(x0 + (w−1)/2, y0 + (h−1)/2)`.

Це не «майже те саме»: ми чекаємо побітового збігу й перевіряємо це через `assert`.

In [ ]:
CROP_ROW, CROP_COL = 60, 90
CROP_HEIGHT, CROP_WIDTH = 64, 48

# наша обрізка: звичайний зріз NumPy у порядку [рядок, стовпець]
our_crop = photo_bgr[CROP_ROW:CROP_ROW + CROP_HEIGHT,
                     CROP_COL:CROP_COL + CROP_WIDTH].copy()

# бібліотечна: розмір і центр у порядку (x, y), центр — посередині цілих пікселів
center_x = CROP_COL + (CROP_WIDTH - 1) / 2.0
center_y = CROP_ROW + (CROP_HEIGHT - 1) / 2.0
library_crop = cv2.getRectSubPix(photo_bgr, (CROP_WIDTH, CROP_HEIGHT), (center_x, center_y))

print("наш зріз          :", our_crop.shape, our_crop.dtype)
print("getRectSubPix     :", library_crop.shape, library_crop.dtype)
print("центр для OpenCV  :", (center_x, center_y))

assert np.array_equal(our_crop, library_crop), "обрізка розійшлася з бібліотечною!"
print("✅ збігається побітово")

## 6 · Скільки шуму викидає інтерполяція

Числа з розділу 5 лекції. Беремо поле чистого випадкового шуму 400 × 400 і зменшуємо його
до 50 × 50 — тобто у 8 разів. На кожен вихідний піксель припадає 64 вхідних.

Стандартне відхилення показує, скільки шуму вижило: чим воно менше, тим більше пікселів
метод фактично усереднив.

In [ ]:
noise_generator = np.random.default_rng(42)
noise_field = noise_generator.integers(0, 256, (400, 400), dtype=np.uint8)

methods = [("INTER_NEAREST", cv2.INTER_NEAREST),
           ("INTER_CUBIC", cv2.INTER_CUBIC),
           ("INTER_LINEAR", cv2.INTER_LINEAR),
           ("INTER_AREA", cv2.INTER_AREA)]

print(f"вхідний шум: std {noise_field.std():.2f}, зменшуємо 400 → 50, тобто у 8 разів")
print()
for name, flag in methods:
    small = cv2.resize(noise_field, (50, 50), interpolation=flag)
    print(f"{name:<14} std {small.std():6.2f}   середнє {small.mean():6.2f}")

print()
print("чесне усереднення 64 чисел дало б std приблизно",
      round(float(noise_field.std()) / 8, 2))

`INTER_AREA` лягає майже точно на теоретичне значення, бо середнє 64 незалежних чисел
коливається рівно у √64 = 8 разів менше, ніж кожне з них.

А тепер дрібниця, яка ловить професіоналів: подивимось на `INTER_LINEAR` при **непарному**
коефіцієнті зменшення.

In [ ]:
# 360 ділиться націло на 2, 3, 4, 5 і 6 — інакше коефіцієнт буде дробовим
# і виродження LINEAR розмиється на півдорозі
square = noise_field[:360, :360]

print("коефіцієнт   LINEAR    AREA")
for factor in range(2, 7):
    side = 360 // factor
    linear = cv2.resize(square, (side, side), interpolation=cv2.INTER_LINEAR)
    area = cv2.resize(square, (side, side), interpolation=cv2.INTER_AREA)
    print(f"    {factor}      {linear.std():6.2f}  {area.std():6.2f}")

print()
print("при непарному коефіцієнті точка вибірки лягає рівно в центр вхідного пікселя,")
print("ваги трьох сусідів обнуляються — і LINEAR читає ОДИН піксель, як NEAREST")

Подивимось на це очима на нашому фото: зменшуємо у 8 разів двома методами й показуємо
результат назад у збільшенні, щоб було видно окремі пікселі.

In [ ]:
small_nearest = cv2.resize(photo_rgb, (40, 30), interpolation=cv2.INTER_NEAREST)
small_area = cv2.resize(photo_rgb, (40, 30), interpolation=cv2.INTER_AREA)

figure, axes = plt.subplots(1, 3, figsize=(11, 3.2))
axes[0].imshow(photo_rgb)
axes[0].set_title("оригінал 320 × 240")
axes[1].imshow(small_nearest, interpolation="nearest")
axes[1].set_title("INTER_NEAREST 40 × 30")
axes[2].imshow(small_area, interpolation="nearest")
axes[2].set_title("INTER_AREA 40 × 30")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

# наскільки два результати взагалі різні
difference = np.abs(small_nearest.astype(np.int32) - small_area.astype(np.int32))
print("середня різниця між двома результатами, рівнів:", round(float(difference.mean()), 2))
print("найбільша різниця, рівнів                     :", int(difference.max()))

## 7 · Перевірка «наше = бібліотечне», раз другий: дзеркалення

Дзеркалення не потребує ані матриці, ані інтерполяції — пікселі просто читаються у
зворотному порядку. Зробимо це зрізом і звіримо з `cv2.flip` по всіх трьох напрямках.

In [ ]:
checks = [
    ("по горизонталі", photo_bgr[:, ::-1], cv2.flip(photo_bgr, 1)),
    ("по вертикалі", photo_bgr[::-1, :], cv2.flip(photo_bgr, 0)),
    ("обидва одразу", photo_bgr[::-1, ::-1], cv2.flip(photo_bgr, -1)),
]

for name, ours, library in checks:
    assert np.array_equal(ours, library), f"дзеркалення {name} розійшлося!"
    print(f"{name:<16} збігається побітово ✅")

print()
# різниця, яка все ж є: зріз віддає ВИД зі зворотним кроком, flip — суцільний масив
print("наш зріз суцільний у памʼяті      :", photo_bgr[:, ::-1].flags["C_CONTIGUOUS"])
print("результат cv2.flip суцільний      :", cv2.flip(photo_bgr, 1).flags["C_CONTIGUOUS"])
print("саме тому деякі функції відмовляться приймати зріз, але приймуть flip")

## 8 · Поворот: скільки пікселів вилітає за кадр

Будуємо матрицю руками й перевіряємо її на центрі — так само, як у лекції. Потім
рахуємо, скільки пікселів кадру виживає при кожному куті.

In [ ]:
center = (IMAGE_WIDTH / 2, IMAGE_HEIGHT / 2)
rotation_matrix = cv2.getRotationMatrix2D(center, 30, 1.0)

print("матриця для 30°:")
print(np.round(rotation_matrix, 4))
print()

# перевіряємо на центрі: він при повороті навколо центра має лишитись на місці
point = np.array([IMAGE_WIDTH / 2, IMAGE_HEIGHT / 2, 1.0])
moved = rotation_matrix @ point
print("центр (160, 120) поїхав у:", np.round(moved, 4))

# і на лівому верхньому куті: там ікс має стати відʼємним
corner = np.array([0.0, 0.0, 1.0])
print("кут   (0, 0)     поїхав у:", np.round(rotation_matrix @ corner, 2))
print("відʼємний ікс означає, що ця точка вилетіла ліворуч за кадр")

In [ ]:
# скільки пікселів кадру виживає: проганяємо через ту саму матрицю білий прямокутник
white_frame = np.full((IMAGE_HEIGHT, IMAGE_WIDTH), 255, np.uint8)
total_pixels = IMAGE_HEIGHT * IMAGE_WIDTH

print("кут   лишилось   частка   полотно, яке вмістило б усе")
for angle in (5, 15, 30, 45, 90):
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    kept_mask = cv2.warpAffine(white_frame, matrix, (IMAGE_WIDTH, IMAGE_HEIGHT),
                               flags=cv2.INTER_NEAREST)
    kept = int((kept_mask > 0).sum())

    radians = np.deg2rad(angle)
    sin_a, cos_a = abs(np.sin(radians)), abs(np.cos(radians))
    new_width = int(IMAGE_HEIGHT * sin_a + IMAGE_WIDTH * cos_a)
    new_height = int(IMAGE_HEIGHT * cos_a + IMAGE_WIDTH * sin_a)

    print(f"{angle:3d}°   {kept:6d}   {kept / total_pixels * 100:5.1f}%   "
          f"{new_width} × {new_height}")

Тепер порахуємо полотно, у якому не зникає нічого, і переконаємось у цьому числом.

In [ ]:
ANGLE = 30
radians = np.deg2rad(ANGLE)
sin_a, cos_a = abs(np.sin(radians)), abs(np.cos(radians))

# формула дає 397.1 × 367.8; додаємо два пікселі запасу, інакше кутові
# ряди губляться на похибці округлення — тому в таблиці вище 397, а тут 399
new_width = int(IMAGE_HEIGHT * sin_a + IMAGE_WIDTH * cos_a) + 2
new_height = int(IMAGE_HEIGHT * cos_a + IMAGE_WIDTH * sin_a) + 2

expanded_matrix = cv2.getRotationMatrix2D(center, ANGLE, 1.0)
# зсуваємо картинку в центр нового полотна: різниця центрів іде в третій стовпець
expanded_matrix[0, 2] += new_width / 2 - IMAGE_WIDTH / 2
expanded_matrix[1, 2] += new_height / 2 - IMAGE_HEIGHT / 2

rotated_same = cv2.warpAffine(photo_rgb,
                              cv2.getRotationMatrix2D(center, ANGLE, 1.0),
                              (IMAGE_WIDTH, IMAGE_HEIGHT))
rotated_expanded = cv2.warpAffine(photo_rgb, expanded_matrix, (new_width, new_height))

kept_same = int((cv2.warpAffine(white_frame,
                                cv2.getRotationMatrix2D(center, ANGLE, 1.0),
                                (IMAGE_WIDTH, IMAGE_HEIGHT),
                                flags=cv2.INTER_NEAREST) > 0).sum())

# точний доказ, що в нове полотно вміщується все: проганяємо через матрицю
# чотири кути кадру й перевіряємо, що жоден не вийшов за межі
corners = np.array([[0.0, 0.0, 1.0], [IMAGE_WIDTH, 0.0, 1.0],
                    [IMAGE_WIDTH, IMAGE_HEIGHT, 1.0], [0.0, IMAGE_HEIGHT, 1.0]]).T
mapped_corners = expanded_matrix @ corners
all_inside = bool(((mapped_corners[0] >= 0) & (mapped_corners[0] <= new_width)
                   & (mapped_corners[1] >= 0) & (mapped_corners[1] <= new_height)).all())

print(f"полотно {IMAGE_WIDTH} × {IMAGE_HEIGHT}: лишилось {kept_same} із {total_pixels} "
      f"({kept_same / total_pixels * 100:.1f}%)")
print(f"полотно {new_width} × {new_height}: усі чотири кути кадру всередині — {all_inside}")
print("кути після повороту:", np.round(mapped_corners.T, 1).tolist())
print(f"ціна: порожніх пікселів у новому полотні "
      f"{(1 - total_pixels / (new_width * new_height)) * 100:.1f}%")

figure, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].imshow(rotated_same)
axes[0].set_title(f"полотно те саме: {kept_same / total_pixels * 100:.1f}% кадру")
axes[1].imshow(rotated_expanded)
axes[1].set_title(f"полотно {new_width} × {new_height}: 100% кадру")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 9 · Межові режими числами

Найменше можливе зображення: масив 3 × 3 зі значеннями від 0 до 8. Додамо рамку в один
піксель і подивимось на верхній рядок результату. Ці чотири рядки знадобляться в темі 04.

In [ ]:
tiny = np.arange(9, dtype=np.uint8).reshape(3, 3)
print("вхідний масив:")
print(tiny)
print()

border_modes = [("BORDER_CONSTANT", cv2.BORDER_CONSTANT),
                ("BORDER_REPLICATE", cv2.BORDER_REPLICATE),
                ("BORDER_REFLECT", cv2.BORDER_REFLECT),
                ("BORDER_REFLECT_101", cv2.BORDER_REFLECT_101),
                ("BORDER_WRAP", cv2.BORDER_WRAP)]

for name, flag in border_modes:
    bordered = cv2.copyMakeBorder(tiny, 1, 1, 1, 1, flag, value=0)
    print(f"{name:<20} верхній рядок: {bordered[0]}")

print()
print("REPLICATE і REFLECT у рамці в один піксель збігаються:")
print("дзеркало REFLECT проходить по зовнішній межі крайнього пікселя,")
print("тому першим відображенням стає він сам")

А ось чому за замовчуванням стоїть саме `REFLECT_101`: подивимось на **перепад через
межу** на справжньому фрагменті фото. Чорна рамка `CONSTANT` сама по собі є різким
перепадом, якого в зображенні не було.

In [ ]:
fragment = cv2.cvtColor(photo_bgr[60:104, 100:144], cv2.COLOR_BGR2GRAY)

print("режим                перепад через межу, рівнів яскравості")
for name, flag in border_modes:
    bordered = cv2.copyMakeBorder(fragment, 1, 1, 1, 1, flag, value=0).astype(np.int32)
    original = fragment.astype(np.int32)
    # верхній рядок рамки проти першого справжнього рядка, і те саме ліворуч
    top_jump = np.abs(bordered[0, 1:-1] - original[0])
    left_jump = np.abs(bordered[1:-1, 0] - original[:, 0])
    print(f"{name:<20} {np.concatenate([top_jump, left_jump]).mean():6.1f}")

## 10 · Маска за кольором і рамка детектора

Переводимо в HSV, беремо зелений відтінок через `inRange`, знаходимо рамку через
`boundingRect` — і малюємо її поверх фото. Це рівно та форма відповіді, яку в темі 02
віддавав детектор: чотири числа.

Зверни увагу на `.copy()` перед малюванням: функції OpenCV пишуть у масив **на місці**.

In [ ]:
hsv = cv2.cvtColor(photo_bgr, cv2.COLOR_BGR2HSV)

# нижня й верхня межі: відтінок зелений, насиченість і яскравість — широко,
# щоб поріг пережив зміну освітлення
green_mask = cv2.inRange(hsv, (40, 80, 60), (85, 255, 255))

box_x, box_y, box_width, box_height = cv2.boundingRect(green_mask)

print("пікселів у маскі      :", int((green_mask > 0).sum()))
print("boundingRect (x, y, w, h):", (box_x, box_y, box_width, box_height))
print("та сама рамка в NumPy    : mask[%d:%d, %d:%d]"
      % (box_y, box_y + box_height, box_x, box_x + box_width))

In [ ]:
# запамʼятовуємо оригінал, щоб потім довести, що він не постраждав
original_backup = photo_bgr.copy()
visual = photo_bgr.copy()     # малюємо ТІЛЬКИ по копії

cv2.rectangle(visual, (box_x, box_y), (box_x + box_width, box_y + box_height),
              (0, 200, 0), 2)
cv2.putText(visual, "indicator", (box_x, box_y - 6), cv2.FONT_HERSHEY_SIMPLEX,
            0.45, (0, 200, 0), 1, cv2.LINE_AA)
cv2.circle(visual, (box_x + box_width // 2, box_y + box_height // 2), 3,
           (0, 0, 255), -1)

print("оригінал не змінився                :",
      np.array_equal(photo_bgr, original_backup))
print("пікселів у візуалізації змінилось   :",
      int((visual != photo_bgr).any(axis=2).sum()))
print()
print("а тепер те саме, але по ВИДУ на оригінал — і без жодного присвоєння:")
danger = photo_bgr.copy()
crop_view = danger[box_y:box_y + box_height, box_x:box_x + box_width]   # вид
cv2.rectangle(crop_view, (0, 0), (box_width - 1, box_height - 1), (0, 200, 0), 2)
print("пікселів «оригіналу» danger змінилось:",
      int((danger != photo_bgr).any(axis=2).sum()), "← функція писала у вид")

figure, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].imshow(green_mask, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
axes[0].set_title("маска inRange")
axes[1].imshow(cv2.cvtColor(visual, cv2.COLOR_BGR2RGB))
axes[1].set_title("рамка детектора поверх фото")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 11 · Три пороги на нерівно освітленій наліпці

Продавець сфотографував наліпку з IMEI на коробці. Треба відокремити темні цифри від
світлого паперу. Наліпку знято при лампі збоку: лівий край майже в тіні, правий
пересвічений.

Мірою беремо **IoU** — той самий перетин, поділений на обʼєднання, що й у темі 02, тільки
рахований по пікселях маски. Частка правильних пікселів тут не годиться: цифри займають
десяту частину наліпки, тож маска «усе папір» дасть майже 90%, нічого не роблячи.

In [ ]:
LABEL_HEIGHT, LABEL_WIDTH = 160, 420

# наліпка: світлий папір, темні цифри
label = np.full((LABEL_HEIGHT, LABEL_WIDTH), 232, np.uint8)
cv2.putText(label, "IMEI 356938035643809", (14, 70),
            cv2.FONT_HERSHEY_SIMPLEX, 1.05, (35,), 2, cv2.LINE_AA)
cv2.putText(label, "S/N  FK2X8ND0QJ7L", (14, 126),
            cv2.FONT_HERSHEY_SIMPLEX, 1.05, (35,), 2, cv2.LINE_AA)

# еталонна маска: що саме є цифрами, ми знаємо точно, бо самі їх намалювали
digits_truth = label < 140

# лампа збоку: множник яскравості росте зліва направо
column_ratio = np.arange(LABEL_WIDTH)[None, :] / LABEL_WIDTH
lamp = 0.17 + 1.45 * column_ratio
lit_label = np.clip(label * lamp, 0, 255)

# трохи шуму матриці, детермінованого — щоб числа збігались у всіх
index = np.arange(LABEL_HEIGHT * LABEL_WIDTH)
grain = (index + 1) * 16807 % 2147483647
grain = grain ^ (grain >> 13)
grain = grain * 48271 % 2147483647
lit_label = np.clip(lit_label + (grain % 9 - 4).reshape(LABEL_HEIGHT, LABEL_WIDTH),
                    0, 255).astype(np.uint8)

ink_values = lit_label[digits_truth]
paper_values = lit_label[~digits_truth]

print("цифри займають, % наліпки                   :",
      round(float(digits_truth.mean()) * 100, 2))
print("маска «усе папір» дала б правильних пікселів:",
      round(float((~digits_truth).mean()) * 100, 2), "%")
print()
print(f"цифри: від {ink_values.min():3d} до {ink_values.max():3d}, "
      f"медіана {int(np.median(ink_values))}")
print(f"папір: від {paper_values.min():3d} до {paper_values.max():3d}, "
      f"медіана {int(np.median(paper_values))}")
print()
print("найтемніший папір має", paper_values.min(),
      "рівень, найсвітліша цифра —", ink_values.max())
print("діапазони перекриваються, тому одного числа, яке їх розділить, НЕ ІСНУЄ")

In [ ]:
def iou_with_digits(mask):
    """Перетин, поділений на обʼєднання, у відсотках — та сама міра, що й у темі 02."""
    predicted = mask > 0
    intersection = np.logical_and(predicted, digits_truth).sum()
    union = np.logical_or(predicted, digits_truth).sum()
    return intersection / max(union, 1) * 100


def accuracy_with_digits(mask):
    return ((mask > 0) == digits_truth).mean() * 100


# найкращий можливий глобальний поріг: перебираємо всі 256 і беремо переможця
best_threshold, best_iou = 0, 0.0
for candidate in range(256):
    _, mask = cv2.threshold(lit_label, candidate, 255, cv2.THRESH_BINARY_INV)
    value = iou_with_digits(mask)
    if value > best_iou:
        best_threshold, best_iou = candidate, value

_, best_mask = cv2.threshold(lit_label, best_threshold, 255, cv2.THRESH_BINARY_INV)
otsu_threshold, otsu_mask = cv2.threshold(lit_label, 0, 255,
                                          cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
adaptive_mask = cv2.adaptiveThreshold(lit_label, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                      cv2.THRESH_BINARY_INV, 31, 6)

# контроль: той самий Оцу, але при рівному освітленні
even_threshold, even_mask = cv2.threshold(label, 0, 255,
                                          cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

print("спосіб                              поріг    IoU     правильних")
print(f"назвати все папером                    —    {0.0:5.1f}%   "
      f"{(~digits_truth).mean() * 100:5.1f}%")
print(f"глобальний, найкращий можливий       {best_threshold:3d}    "
      f"{best_iou:5.1f}%   {accuracy_with_digits(best_mask):5.1f}%")
print(f"глобальний, Оцу                      {otsu_threshold:3.0f}    "
      f"{iou_with_digits(otsu_mask):5.1f}%   {accuracy_with_digits(otsu_mask):5.1f}%")
print(f"адаптивний, блок 31                    —    "
      f"{iou_with_digits(adaptive_mask):5.1f}%   {accuracy_with_digits(adaptive_mask):5.1f}%")
print(f"Оцу при рівному світлі               {even_threshold:3.0f}    "
      f"{iou_with_digits(even_mask):5.1f}%   {accuracy_with_digits(even_mask):5.1f}%")

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(11, 4.4))
panels = [(lit_label, "наліпка: лампа світить справа"),
          (best_mask, f"найкращий глобальний, t={best_threshold}: "
                      f"IoU {best_iou:.1f}%"),
          (otsu_mask, f"Оцу, t={otsu_threshold:.0f}: "
                      f"IoU {iou_with_digits(otsu_mask):.1f}%"),
          (adaptive_mask, f"адаптивний, блок 31: "
                          f"IoU {iou_with_digits(adaptive_mask):.1f}%")]

for axis, (image, title) in zip(axes.ravel(), panels):
    axis.imshow(image, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    axis.set_title(title, fontsize=10)
    axis.axis("off")
plt.tight_layout()
plt.show()

print("Оцу поділив кадр на темну й світлу половини — але ця межа проходить по лампі,")
print("а не по цифрах. Адаптивний рахує свій поріг для кожного пікселя по його околу.")

## 12 · Скільки це коштує на кадрі 1080p

Останнє: поміряємо час кожної операції на своїй машині. Числа в тебе будуть інші, ніж у
лекції, — залежить від процесора й від того, чим він зайнятий. Стійким лишається
**співвідношення**.

Міряємо в один потік (`cv2.setNumThreads(1)`) і беремо **мінімум** із прогонів: мінімум
менше залежить від сторонніх процесів, ніж середнє.

In [ ]:
cv2.setNumThreads(1)          # інакше результат залежить від того, скільки ядер вільні

frame = np.random.default_rng(0).integers(0, 256, (1080, 1920, 3), dtype=np.uint8)
warp_matrix = cv2.getRotationMatrix2D((960, 540), 12, 1.0)

operations = [
    ("img.copy()", lambda: frame.copy()),
    ("resize → 960 × 540, NEAREST",
     lambda: cv2.resize(frame, (960, 540), interpolation=cv2.INTER_NEAREST)),
    ("flip по горизонталі", lambda: cv2.flip(frame, 1)),
    ("cvtColor BGR → GRAY", lambda: cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)),
    ("resize → 960 × 540, AREA",
     lambda: cv2.resize(frame, (960, 540), interpolation=cv2.INTER_AREA)),
    ("cvtColor BGR → HSV", lambda: cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)),
    ("warpAffine, поворот на 12°",
     lambda: cv2.warpAffine(frame, warp_matrix, (1920, 1080))),
]

FRAME_BUDGET_MS = 1000 / 25    # 25 кадрів за секунду = 40 мс на кадр
RUNS = 30

print(f"кадр 1920 × 1080 × 3, один потік, мінімум із {RUNS} прогонів")
print(f"бюджет одного кадру при 25 к/с: {FRAME_BUDGET_MS:.0f} мс")
print()
print("операція                       мс     частка кадру")

timings = {}
for name, action in operations:
    action()                                     # прогрів: перший виклик завжди довший
    samples = []
    for _ in range(RUNS):
        started = time.perf_counter()
        action()
        samples.append((time.perf_counter() - started) * 1000)
    best = min(samples)
    timings[name] = best
    print(f"{name:<30} {best:5.2f}   {best / FRAME_BUDGET_MS * 100:5.1f}%")

In [ ]:
preprocess = (timings["img.copy()"]
              + timings["resize → 960 × 540, AREA"]
              + timings["cvtColor BGR → GRAY"])
with_hsv = preprocess + timings["cvtColor BGR → HSV"]
with_warp = with_hsv + timings["warpAffine, поворот на 12°"]

print("типовий передобробіток (копія + зменшення + сірий):")
print(f"  {preprocess:5.2f} мс = {preprocess / FRAME_BUDGET_MS * 100:4.1f}% бюджету, "
      f"на модель лишається {FRAME_BUDGET_MS - preprocess:5.2f} мс")
print("той самий, але з переведенням у HSV:")
print(f"  {with_hsv:5.2f} мс = {with_hsv / FRAME_BUDGET_MS * 100:4.1f}% бюджету, "
      f"лишається {FRAME_BUDGET_MS - with_hsv:5.2f} мс")
print("із вирівнюванням повороту:")
print(f"  {with_warp:5.2f} мс = {with_warp / FRAME_BUDGET_MS * 100:4.1f}% бюджету, "
      f"лишається {FRAME_BUDGET_MS - with_warp:5.2f} мс")
print()
print("співвідношення, яке не залежить від машини:")
print(f"  HSV дорожчий за сірий у "
      f"{timings['cvtColor BGR → HSV'] / timings['cvtColor BGR → GRAY']:.1f} раза")
print(f"  поворот дорожчий за сірий у "
      f"{timings['warpAffine, поворот на 12°'] / timings['cvtColor BGR → GRAY']:.1f} раза")

## 13 · Прибираємо за собою

Усі файли жили в тимчасовій теці. Видаляємо її разом із вмістом.

In [ ]:
files_before = sorted(os.listdir(WORK_DIR))
shutil.rmtree(WORK_DIR)

print("було у тимчасовій теці:", files_before)
print("тека існує після прибирання:", os.path.exists(WORK_DIR))

## Завдання

Повний опис із критеріями «зроблено» — у [homework.html](homework.html).

**🟢 Рівень 1.** Згенеруй власне синтетичне зображення (не телефон — наприклад, «фото
цінника»: світлий прямокутник із темною смугою) і прожени його через увесь ланцюжок:
запис у PNG і JPEG, обрізка зрізом, зменшення трьома інтерполяціями, поворот на 20°.
Для кожного кроку надрукуй, що саме він викинув.

**🟡 Рівень 2.** Напиши функцію `letterbox(image, target_size)`, яка вписує зображення в
квадрат заданого розміру зі збереженням пропорцій і доповнює поля сірим. Перевір на
трьох різних вхідних розмірах, що результат завжди має форму `(target, target, 3)` і що
співвідношення сторін предмета не змінилось.

**🔴 Рівень 3.** Реалізуй `INTER_AREA` з нуля на NumPy для цілого коефіцієнта зменшення й
доведи через `np.allclose`, що твій результат збігається з `cv2.resize`. Потім знайди
коефіцієнт, при якому збіг ламається, і поясни чому.